In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense, Dropout,
                                     BatchNormalization, Reshape,
                                     MultiHeadAttention, LayerNormalization)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import tensorflow.keras.backend as K

In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

img_size = (224,224)
batch_size = 16

In [ ]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    brightness_range=[0.7,1.3],
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

train_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

In [ ]:
def focal_loss(gamma=2., alpha=0.75):
    def loss(y_true, y_pred):
        y_pred = K.clip(y_pred, 1e-7, 1-1e-7)
        pt = y_true*y_pred + (1-y_true)*(1-y_pred)
        return -K.mean(alpha * K.pow(1-pt, gamma) * K.log(pt))
    return loss

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze layers (Phase 1)
for layer in base_model.layers[:-30]:
    layer.trainable = False

x = base_model.output   # (7,7,1280)

# Convert to sequence
x = Reshape((49, 1280))(x)

# Transformer Block
attn = MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
x = LayerNormalization()(x + attn)

# Feed Forward
ff = Dense(256, activation='relu')(x)
ff = Dense(1280)(ff)
x = LayerNormalization()(x + ff)

# Back to spatial
x = Reshape((7,7,1280))(x)

# Pooling
x = GlobalAveragePooling2D()(x)

# Dense block
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(128, activation='relu')(x)

output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',   
    metrics=['accuracy']
)

model.summary()

In [ ]:
class_weights = {0:1.0, 1:2.0}

In [ ]:
early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early, lr_reduce]
)

In [ ]:
# Unfreeze ALL layers
for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=Adam(1e-5),
    loss=focal_loss(),
    metrics=['accuracy']
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early, lr_reduce]
)

In [ ]:
def tta(model, data, n=5):
    preds = []
    for _ in range(n):
        data.reset()
        preds.append(model.predict(data))
    return np.mean(preds, axis=0)

y_pred_prob = tta(model, val_data)
y_true = val_data.classes

In [ ]:
best_f1 = 0
best_t = 0.5

for t in np.arange(0.4, 0.6, 0.01):
    y_pred = (y_pred_prob > t).astype(int)
    f1 = f1_score(y_true, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best Threshold:", best_t)

In [ ]:
y_pred = (y_pred_prob > best_t).astype(int)

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.show()